# Sesion 2: Contenedorizacion con Docker

La API que construimos en la session pasada todavia depende de que la computadora donde corra tenga instalado exactamente Python, FastAPI, scikit-learn y joblib en las versiones correctas. Si le pasas tu carpeta de proyecto a un companero, o la subes a un servidor, no hay garantia de que funcione igual.

Docker resuelve este problema empaquetando la aplicacion junto con **todo su entorno** (version de Python, librerias, sistema de archivos) en una unidad portatil llamada **imagen**, de la cual se pueden crear **container** en ejecucion.

**Importante antes de empezar:** no se puede utilizar Docker desde Google Colab. En esta notebook vamos a preparar todos los archivos necesarios, y vamos a hacer una demostracion en vivo construyendo y corriendo el contenedor en una computadora real. Tendrian que repetir el mismo proceso vos mismo, en su propia computadora, con Docker Desktop instalado.

Hoy vamos a:

1. Entender que es una imagen, un contenedor y un Dockerfile.
2. Preparar el archivo `requirements.txt`.
3. Escribir el `Dockerfile` para nuestra API de scoring crediticio.
4. Escribir un `.dockerignore`.
5. Resolver un mini reto.
6. Ver la checklist de comandos para correrlo en una computadora real.

## El problema que resuelve Docker

La frase: *"en mi computadora funciona"*. Pasa todo el tiempo: un modelo que corre perfecto en tu laptop falla en el servidor porque:

- El servidor tiene una version distinta de Python.
- Falta una libreria, o esta en otra version.
- El sistema operativo maneja las rutas de archivos de otra forma.

Docker soluciona esto porque **no dependemos del entorno de la computadora anfitriona**. La imagen incluye su propio sistema de archivos minimo, con la version exacta de Python y las librerias que nosotros definimos. Si la imagen corre en tu laptop, corre igual en el servidor de la empresa, o en la nube.

## Conceptos clave

- **Imagen (image)**: es como una "foto" congelada de todo lo que necesita la aplicacion para correr: sistema base, librerias instaladas, codigo, y el modelo entrenado. Se construye una sola vez a partir de una receta.
- **Dockerfile**: es esa receta. Un archivo de texto con instrucciones paso a paso de como armar la imagen (que version de Python usar, que instalar, que archivos copiar).
- **Contenedor (container)**: es una instancia en ejecucion de una imagen. De la misma imagen se pueden levantar varios contenedores al mismo tiempo, igual que de una misma receta se pueden hacer varias tortas.

Una buena analogia: el Dockerfile es la receta de cocina, la imagen es la torta ya horneada y lista, y el contenedor es la porcion que efectivamente servis y comes.

## Paso 1: El archivo requirements.txt

Antes de escribir el Dockerfile necesitamos la lista exacta de librerias que la API necesita para funcionar. Docker va a usar este archivo para instalar todo dentro de la imagen.

In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
joblib
pydantic

Writing requirements.txt


## Paso 2: Escribir el Dockerfile

El Dockerfile se lee de arriba hacia abajo, y cada linea es una instruccion que agrega una "capa" a la imagen. Vamos linea por linea:

- `FROM python:3.9-slim`: define la imagen base. Usamos una version "slim" (liviana) para no cargar con cosas que no necesitamos.  

- `WORKDIR /app`: crea y establece `/app` como el directorio de trabajo dentro del contenedor. Todo lo que hagamos despues pasa ahi adentro.  

- `COPY requirements.txt .`: copia solo el archivo de dependencias primero (esto es una buena practica: si el codigo cambia pero las dependencias no, Docker puede reusar esta capa y construir mas rapido).  

- `RUN pip install --no-cache-dir -r requirements.txt`: instala las librerias dentro de la imagen.  

(```--no-cache-dir``` ayuda a reducir el tamaño de la imagen Docker al evitar que pip conserve archivos innecesarios en su cache.)

- `COPY . .`: copia el resto del proyecto (el `main.py`, el `model.pkl`, etc.).  

- `EXPOSE 8000`: documenta que el contenedor va a escuchar en el puerto 8000.  

- `CMD [...]`: el comando que se ejecuta automaticamente cuando arranca el contenedor. En nuestro caso, levantar el servidor `uvicorn`.  

Un detalle que suele causar errores: dentro del comando de `uvicorn` usamos `--host 0.0.0.0` y no `127.0.0.1`. Si usaramos `127.0.0.1`, la API solo aceptaria conexiones desde dentro del propio contenedor, y no podriamos acceder a ella desde afuera.

In [ ]:
%%writefile Dockerfile
FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


Elementos dentro de CMD:
```
CMD [
    "uvicorn",       # Servidor que ejecuta la aplicación FastAPI
    "main:app",      # Archivo main.py y objeto FastAPI llamado app
    "--host",        # Define la dirección donde escucha el servidor
    "0.0.0.0",       # Permite conexiones desde fuera del contenedor
    "--port",        # Define el puerto que utiliza el servidor
    "8000"           # Puerto donde se ejecuta la API
]
```

## Paso 3: El archivo .dockerignore

Asi como usamos `.gitignore` para que Git no suba archivos innecesarios, usamos `.dockerignore` para que Docker no copie dentro de la imagen carpetas pesadas o irrelevantes, como el historial de Git o los archivos temporales de Python. Esto mantiene la imagen liviana y la construccion mas rapida.

In [ ]:
%%writefile .dockerignore
__pycache__/
*.pyc
.git
.ipynb_checkpoints
venv/

Writing .dockerignore


## Buenas practicas y errores comunes

Buenas practicas:

- Usar imagenes base livianas (`python:3.9-slim` en vez de la imagen completa).
- Mantener `requirements.txt` actualizado y con solo lo necesario.
- No copiar datasets gigantes dentro de la imagen; si se necesitan, se manejan aparte con volumenes.

Errores comunes:

- **Imagenes muy pesadas**: olvidar `--no-cache-dir` en `pip install`, o usar una imagen base completa en vez de `slim`, puede hacer que la imagen pese varios GB de mas.
- **Hardcodear 127.0.0.1**: como vimos arriba, hace que la API sea inaccesible desde afuera del contenedor.
- **No manejar excepciones**: si el modelo falla al predecir, la API deberia devolver un error controlado (por ejemplo, codigo 500 con un mensaje claro) en vez de colapsar sin explicacion. Vamos a mejorar esto en la Sesion 3.
- **Olvidar el .dockerignore**: copiar `.git` o `__pycache__` hace la imagen mas pesada y la construccion mas lenta de lo necesario.

## Practicamos

Escribamos un Dockerfile para otra aplicacion, distinta a la nuestra. Esta aplicacion:

- Usa `pandas` y `scikit-learn`.
- Se ejecuta a partir de un archivo llamado `api_modelo.py` (no `main.py`).
- Escucha en el puerto `5000` en vez de `8000`.

Escribi el Dockerfile completo para este caso. Fijate bien que instrucciones cambian respecto al Dockerfile que ya escribimos, y cuales quedan igual.

In [ ]:
%%writefile Dockerfile_reto
FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["uvicorn", "api_modelo:app", "--host", "0.0.0.0", "--port", "5000"]

Writing Dockerfile_reto


In [ ]:
import os
os.remove("Dockerfile_reto")

---

## Hacerlo en vivo tu computadora

Los archivos que preparamos en nuestro notebooks (`main.py`, `model.pkl`, `requirements.txt`, `Dockerfile`, `.dockerignore`) son los que necesitas para construir y correr el contenedor en una computadora real.

#### Generar de nuevo el model.pkl y main.py si es necesario :

**Generar ``` model.pkl ```:**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 500

ingresos = np.random.uniform(300, 5000, n)
edad = np.random.randint(18, 70, n)
historial_deuda = np.random.randint(0, 2, n)

score = (ingresos * 0.0006) - (historial_deuda * 1.5) + (edad * 0.01) + np.random.normal(0, 0.5, n)
aprobado = (score > score.mean()).astype(int)

df = pd.DataFrame({"ingresos": ingresos, "edad": edad, "historial_deuda": historial_deuda, "aprobado": aprobado})

X = df[["ingresos", "edad", "historial_deuda"]]
y = df["aprobado"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)
print("Accuracy:", modelo.score(X_test, y_test))

# Guardar el modelo entrenado en un archivo binario (.pkl)
import joblib
joblib.dump(modelo, "model.pkl")

Accuracy: 0.91


['model.pkl']

**Generar ``` main.py ```:**

In [ ]:
%%writefile main.py

import joblib
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="API de Scoring Crediticio")

modelo = joblib.load("model.pkl")

class ClienteInput(BaseModel):
    ingresos: float
    edad: int
    historial_deuda: int

class VentasInput(BaseModel):
    ventas_mensuales: float

@app.get("/health")
def health():
    return {"status": "online"}

@app.post("/predict")
def predict(cliente: ClienteInput):
    datos = [[cliente.ingresos, cliente.edad, cliente.historial_deuda]]
    prediccion = modelo.predict(datos)[0]
    decision = "Aprobado" if prediccion == 1 else "Rechazado"

    return {
        "edad": cliente.edad,
        "decision": decision
    }

@app.post("/proyeccion")
def proyeccion(ventas: VentasInput):
    comision = ventas.ventas_mensuales * 0.10

    return {
        "proyeccion_comision": comision
    }

Writing main.py


## Docker desde el Terminal

0. Instalar Docker (https://www.docker.com/products/docker-desktop/).  

1. Descarga los archivos generados en este notebook a una carpeta en tu computadora (podes usar el panel de archivos de Colab, boton derecho > Descargar). Llamamos la carpeta ``` credit_scoring_api ```.  
Para hacerlo puede ejecutar este codigo :  

In [ ]:
import zipfile  # Importamos zipfile para crear el archivo ZIP
from google.colab import files  # Importamos files para descargar archivos desde Colab

files_to_download = [  # Definimos los archivos que queremos descargar
    "main.py",
    "model.pkl",
    "requirements.txt",
    "Dockerfile",
    ".dockerignore"
]

with zipfile.ZipFile("credit_scoring_api.zip", "w") as zip_file:  # Creamos el archivo ZIP
    for file in files_to_download:  # Recorremos cada archivo de la lista
        zip_file.write(file)  # Agregamos el archivo al ZIP

files.download("credit_scoring_api.zip")  # Descargamos el ZIP a nuestro computador

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2. Abre una terminal y navega hasta esa carpeta.  

3. Construye la imagen:
   `docker build -t ml-api-app .`  
   >Es equivalente a decir al terminal: "Construir una imagen Docker a partir de la carpeta actual y asignarle el nombre 'ml-api-app'")  <br>
   >
   >Ahora tenemos una Docker Image almacenada localmente.  
   >Se puede verificar con ```docker images```en el Terminal, o directament en Docker App.

4. Corre el contenedor a partir de la imagen:
   `docker run -p 8000:8000 ml-api-app`    

>Esta instrucción significa :
>
>```docker run``` → crea y ejecuta un contenedor a partir de una imagen.
>```ml-api-app``` → es la imagen que construimos anteriormente con docker build.  
>```-p 8000:8000``` → conecta el puerto de nuestra computadora con el puerto del contenedor.   
  

5. Abre en el navegador `http://localhost:8000/docs`.  
> ```localhost``` significa nuestra propia computadora.
> El puerto 8000 es el puerto que acabamos de conectar mediante: ```-p 8000:8000```

Deberias ver la documentacion interactiva de Swagger UI, generada automaticamente por FastAPI, donde podes probar el endpoint `/predict` directamente desde el navegador.

Si todo funciona, acabas de correr tu primer modelo de Machine Learning dentro de un contenedor Docker.

>Nota :  
>En el ejercicio anterior, FastAPI se ejecutaba directamente en nuestro entorno de Python. Ahora la misma API está empaquetada y ejecutándose dentro de un contenedor Docker.  
```
ANTES
Python environment → FastAPI → Model

AHORA
Docker Container → Python → FastAPI → Model
```

## Docker desde el Desktop

Docker Desktop permite hacer gran parte de este proceso de manera gráfica, sin necesidad de escribir todos los comandos en la terminal.

Sin embargo, hay una diferencia importante:

Docker Desktop permite administrar imágenes y contenedores mediante una interfaz gráfica, pero para construir una imagen a partir de un Dockerfile, dependiendo de la versión de Docker Desktop, puede seguir siendo más sencillo utilizar la terminal.

Para nuestro proyecto de FastAPI + Machine Learning, podemos acercarnos bastante a un flujo de trabajo de "point and click".

1. Estructura de archivos

Debemos tener todos los archivos del proyecto dentro de una misma carpeta:
```
credit_scoring_api/
├── main.py
├── model.pkl
├── requirements.txt
├── Dockerfile
└── .dockerignore
```


2. Abrir Docker Desktop

Abre **Docker Desktop**.

Desde Docker Desktop podemos administrar visualmente:

* Docker Images
* Docker Containers
* Logs
* Puertos
* Estado de los contenedores
* Terminales dentro de los contenedores

Esto nos permite visualizar lo que normalmente haríamos utilizando comandos en la terminal.


3. Construir la Docker Image


Aunque **Docker Desktop** permite administrar visualmente nuestras imágenes y contenedores, la construcción de una nueva imagen a partir de un `Dockerfile` se realiza de forma más sencilla desde la **terminal**.

Para construir nuestra imagen debemos:

1. Abrir una terminal.
2. Navegar hasta la carpeta que contiene nuestro proyecto

3. Ejecutar:

```bash
docker build -t ml-api-app .
```

Este comando le indica a Docker:

* `docker build` → construir una nueva imagen.
* `-t ml-api-app` → asignarle el nombre `ml-api-app`.
* `.` → utilizar la carpeta actual como contexto de construcción y buscar allí el `Dockerfile`.

Una vez terminada la construcción, podemos regresar a **Docker Desktop → Images**, donde veremos nuestra nueva imagen `ml-api-app`.



4. Ejecutar el contenedor desde Docker Desktop

Una vez creada la imagen:

1. Ve a **Images** en Docker Desktop.
2. Busca la imagen `ml-api-app`.
3. Selecciona **Run**.
4. Configura el puerto.

Necesitamos conectar:

```
Nuestra machina                    Docker container

8000   ──────────→   :8000/tcp
```

Esta configuración es equivalente al comando de terminal:

```bash
docker run -p 8000:8000 ml-api-app
```

Después de hacer clic en **Run**, Docker Desktop creará y ejecutará un nuevo contenedor a partir de nuestra imagen.


5. Ver el contenedor ejecutándose

Ahora podemos ir a la sección **Containers** de Docker Desktop.

Deberíamos ver nuestro contenedor ejecutándose, junto con información como:

```text
credit-scoring-api
● Running

Port:
8000 → 8000
```

Desde esta sección podemos realizar varias acciones utilizando la interfaz gráfica:

* Iniciar el contenedor
* Detenerlo
* Reiniciarlo
* Eliminarlo
* Ver los logs
* Inspeccionar su configuración
* Abrir una terminal dentro del contenedor


6. Probar nuestra API

Una vez que el contenedor esté ejecutándose, abrimos nuestro navegador y vamos a:

```text
http://localhost:8000/docs
```

Deberíamos ver la interfaz de **Swagger UI** generada automáticamente por FastAPI.

Desde ahí podemos probar nuestro endpoint:

```text
/predict
```

Por lo tanto, si todo funciona correctamente, tendremos nuestro modelo de Machine Learning ejecutándose dentro de un contenedor Docker.
